# EmergencyFlow AI: XGBoost travel-time model (prototype)

**What this notebook does:** trains an **XGBoost** model on our own recorded simulation runs to predict
**how many seconds the ambulance needs on a road**, from live traffic data (queue, cars, speed, signal).

**Status (honest):** this is the **next step** after v1. The running system (v1) plans routes with a
time-dependent shortest-path search and a hand-written travel-time formula. This notebook shows the
learned model that would replace that formula. It is a prototype, not yet wired into the app.

## Why we train on simulation

Training an emergency system on real ambulances is unsafe and slow, so the industry trains and tests in simulation first:

* **Waymo** has driven **10+ billion miles in simulation** (2019), and **20+ billion** since, against tens of millions of real miles.
* **Tesla** collects **billions of real miles** from its customer fleet (1 billion on Autopilot by 2018; several billion on FSD since) and replays hard cases in simulation.
* **SUMO**, the simulator we use, is an open-source traffic simulator developed by the German Aerospace Center (DLR) and used in transport research worldwide.

**Our data:** 560 paired simulation runs on a 16-junction city grid, of which **88 were recorded in full** (ambulance position 5× per second, every signal change, queues on every road every 5 s). Every recording was re-checked against the original result (17 metrics, all identical).

In [ ]:
# Libraries: xgboost (the model), pandas (tables), scikit-learn (splitting + scoring).
# !pip -q install xgboost        # (already installed on Colab; uncomment if not)
import glob, gzip, json, os, subprocess
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error

## 1. Get our recorded runs
Downloads only the recordings (a few MB) from our public GitHub repository.

In [ ]:
# Our recorded simulation runs live on GitHub (experiments/telemetry/*.json.gz).
# Download ONLY that folder (fast, no 3D models).
REPO = "https://github.com/SamarthKale/Ambulance-Router-evp-digital-twin.git"
if not os.path.exists("repo"):
    subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "repo"], check=True)
    subprocess.run(["git", "-C", "repo", "sparse-checkout", "set", "experiments/telemetry"], check=True)
files = sorted(glob.glob("repo/experiments/telemetry/*/*.json.gz"))
print(len(files), "recorded runs")   # 88: 8 strategies x several traffic seeds and demand levels

## 2. Build the training table
One **row** = one road the ambulance drove in one run.
* **Inputs (features):** what the system knew when the ambulance entered the road.
* **Output (target):** the seconds it actually took on that road.

In [ ]:
# One ROW = one road the ambulance drove along in one run.
# FEATURES = what the system knew when the ambulance ENTERED the road (live SUMO data).
# TARGET   = how many seconds the ambulance actually needed for that road.
def rows_from_run(path):
    run = json.load(gzip.open(path))
    arm = os.path.basename(os.path.dirname(path))            # e.g. "basic_static"
    scale = float(path.split("_x")[1].split("_")[0])          # traffic demand, e.g. 1.5
    roads = run["ambulance_edges"]                            # road names used by the track
    track = run["ambulance"]                                  # [t, x, y, angle, speed, road, lane] at 5 Hz
    edge_ids = run["edges"]["ids"]                            # per-road queue samples, every 5 s
    samples = run["edges"]["samples"]
    status = run["status"]                                    # [t, eta, metres left, next signal, its colour] at 1 Hz

    # group the track into consecutive stretches on the same road
    stretches, start = [], 0
    for i in range(1, len(track) + 1):
        if i == len(track) or track[i][5] != track[start][5]:
            stretches.append((start, i - 1)); start = i
    rows = []
    for k, (a, b) in enumerate(stretches[:-1]):               # the last road ends at the hospital: skip it
        road = roads[track[a][5]]
        if road.startswith(":"):                              # ":" = inside a junction, not a road
            continue
        t_in, t_out = track[a][0], track[b + 1][0]            # entry time, first sample after leaving
        s = max([x for x in samples if x["t"] <= t_in] or samples[:1], key=lambda x: x["t"])
        j = edge_ids.index(road)
        st = max([x for x in status if x[0] <= t_in] or status[:1], key=lambda x: x[0])
        rows.append({
            "run": path,
            "queue_halted": s["halting"][j],                  # cars stopped on this road
            "vehicles": s["vehicles"][j],                     # cars on this road
            "mean_speed": s["speed"][j],                      # their mean speed (m/s)
            "entry_speed": track[a][4],                       # ambulance speed when it entered
            "signal_green": 1 if (st[4] or "r") in "Gg" else 0,  # its next signal green on entry?
            "edge_road": 1 if road[0] in "nswe" else 0,       # map-edge roads are 200 m, inner 250 m
            "preemption": 1 if arm.split("_")[0] in ("basic", "coord") else 0,  # signal priority on?
            "coord": 1 if arm.startswith("coord") else 0,
            "crosses_red": 1 if arm.startswith("off_realistic") else 0,
            "demand": scale,
            "seconds_on_road": t_out - t_in,                  # TARGET
        })
    return rows

data = pd.DataFrame([r for f in files for r in rows_from_run(f)])
print(data.shape)
data.head()

## 3. Train XGBoost
XGBoost = many small decision trees added one after another; each new tree corrects the errors of the ones before.
We split **by run**, so the test missions are ones the model has never seen.

In [ ]:
# Split by RUN (never the same mission in train and test), then train XGBoost.
features = [c for c in data.columns if c not in ("run", "seconds_on_road")]
split = GroupShuffleSplit(test_size=0.25, random_state=42)
train_idx, test_idx = next(split.split(data, groups=data["run"]))
train, test = data.iloc[train_idx], data.iloc[test_idx]

model = xgb.XGBRegressor(
    n_estimators=200,      # 200 small trees, each correcting the previous ones' errors
    max_depth=4,           # each tree asks at most 4 yes/no questions
    learning_rate=0.05,    # how much each new tree is allowed to correct
    subsample=0.9,
    random_state=42,
)
model.fit(train[features], train["seconds_on_road"])

## 4. How good is it?

In [ ]:
# Compare with a naive guess (always the training average) on runs the model never saw.
pred = model.predict(test[features])
naive = np.full(len(test), train["seconds_on_road"].mean())
print(f"XGBoost error : {mean_absolute_error(test['seconds_on_road'], pred):.1f} s per road")
print(f"Naive average : {mean_absolute_error(test['seconds_on_road'], naive):.1f} s per road")

## 5. What does the model look at?

In [ ]:
# Which inputs mattered most?
pd.Series(model.feature_importances_, index=features).sort_values().plot.barh(title="What the model looks at")

## 6. Use it like the router would

In [ ]:
# Ask the model about one road, the way the router would during a live mission.
example = pd.DataFrame([{
    "queue_halted": 6, "vehicles": 12, "mean_speed": 3.0, "entry_speed": 15.0,
    "signal_green": 0, "edge_road": 0, "preemption": 0, "coord": 0, "crosses_red": 0, "demand": 1.5,
}])
print(f"Predicted time on this road: {model.predict(example[features])[0]:.1f} s")
example["preemption"] = 1
print(f"Same road with preemption : {model.predict(example[features])[0]:.1f} s")

## 7. Save the model

In [ ]:
# Save the model as a small JSON file: this is what the backend would load.
model.save_model("eta_model.json")
print(os.path.getsize("eta_model.json") // 1024, "KB")

## 8. How this joins our current code

```text
SUMO (traffic truth) ──TraCI──► engine.py ──live road data──► ai/routing.py (route search)
                                                                │  cost of each road =
                                                                │  drive + queue + signal  ◄── v1: hand-written formula
                                                                │                         ◄── next: XGBoost prediction (this notebook)
                                                                ▼
                                    safety/controller.py  (UNCHANGED: checks every signal change)
```

1. **Training (offline, laptop/Colab):** this notebook → `eta_model.json` (about 300 KB).
2. **Backend:** a new file `backend/ai/learned_eta.py` loads `eta_model.json` once, and has one function
   `predict_road_seconds(features) -> float`.
3. **Routing:** in `backend/ai/routing.py`, the cost of a road (today: drive time + queue + signal wait from a formula)
   calls `predict_road_seconds(...)` instead. The inputs already exist there: the engine already reads queue, cars and
   speed for every road from SUMO every second.
4. **Safety:** nothing changes in `backend/safety/`. The model only changes **which route is suggested**; it can never
   switch a signal. Every signal change still goes through the safety controller and the independent monitor.
5. **Proof:** re-run the same paired experiments (`scripts/run_experiments.py`) with and without the model and compare
   predicted vs. measured times, exactly as we did for v1.

**On a microcontroller (optional demo):** the trained trees can be exported to plain C (tool: `m2cgen`) and run on an
ESP32/NodeMCU in milliseconds. In a real deployment the model stays on the central server.

In [ ]:
# How the backend would use the saved model (sketch of backend/ai/learned_eta.py):
#
#   import xgboost as xgb
#   _model = xgb.XGBRegressor(); _model.load_model("eta_model.json")      # loaded once at start-up
#
#   def predict_road_seconds(queue_halted, vehicles, mean_speed, entry_speed, signal_green,
#                            edge_road, preemption, coord, crosses_red, demand) -> float:
#       row = [[queue_halted, vehicles, mean_speed, entry_speed, signal_green,
#               edge_road, preemption, coord, crosses_red, demand]]
#       return float(_model.predict(row)[0])
#
# ai/routing.py then uses predict_road_seconds(...) as the cost of each road in its
# shortest-path search. The safety layer (backend/safety/) is untouched.
loaded = xgb.XGBRegressor()
loaded.load_model("eta_model.json")
print("reloaded model predicts:", round(float(loaded.predict(example[features])[0]), 1), "s")